In [2]:
# example code from chatgpt

from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset

# Load the dataset
dataset = load_dataset('path_to_your_dataset')

# Initialize the tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Initialize the model
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.resize_token_embeddings(len(tokenizer))

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
)

# Fine-tune the model
trainer.train()


DatasetNotFoundError: Dataset 'path_to_your_dataset' doesn't exist on the Hub or cannot be accessed. If the dataset is private or gated, make sure to log in with `huggingface-cli login` or visit the dataset page at https://huggingface.co/datasets/path_to_your_dataset to ask for access.

### preprocess data: using tokenizer

In [3]:
from PolymerSmilesTokenization import PolymerSmilesTokenizer
from torch.utils.data import Dataset, DataLoader
import pandas as pd

class LoadPretrainData(Dataset):
    def __init__(self, tokenizer, dataset, blocksize):
        self.tokenizer = tokenizer
        self.blocksize = blocksize
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, i):
        smiles = self.dataset[i][0]
        encoding = self.tokenizer(
            str(smiles),
            add_special_tokens=True,
            max_length=self.blocksize,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
        }


### Fine-Tuning GPT-2

In [4]:
# load the pre-trained GPT-2 model and tokenizer from Hugging Face

from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments

model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Add special tokens if necessary
#special_tokens_dict = {'additional_special_tokens': ['<polymer>', '</polymer>']}
#num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
#model.resize_token_embeddings(len(tokenizer))


Downloading: 100%|██████████| 665/665 [00:00<00:00, 4.10MB/s]
Downloading: 100%|██████████| 523M/523M [00:32<00:00, 17.1MB/s] 
Downloading: 100%|██████████| 0.99M/0.99M [00:00<00:00, 20.1MB/s]
Downloading: 100%|██████████| 446k/446k [00:00<00:00, 9.17MB/s]
Downloading: 100%|██████████| 26.0/26.0 [00:00<00:00, 187kB/s]


In [ ]:
# Prepare data and define the training loop:

# Load your dataset
dataset = pd.read_csv('path_to_your_dataset.csv')
train_data = LoadPretrainData(tokenizer, dataset, blocksize=512)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
)

trainer.train()


### Generate polymer strings

In [ ]:
input_properties = "desired properties in specific format"
input_ids = tokenizer.encode(input_properties, return_tensors='pt')
generated_text_samples = model.generate(input_ids, max_length=512, num_return_sequences=3)
for sample in generated_text_samples:
    print(tokenizer.decode(sample, skip_special_tokens=True))
